# Field validation — `energetics` (DEPTH pipeline)

| | |
|---|---|
| Subset | `energetics` |
| Pipeline | DEPTH |
| Timestep | 2012-11-09 12:00:00 |
| Domain | one 720 × 720 × 51 tile (≈1400 × 1400 km), set in Section 1 |
| Depth levels | `sfc`, `z25m`, `mld`, `mld_mean` |
| Data | computed on the fly from `s3://dbof/LLC4320_RAW/DEPTH/` |
| Plan | `prompts/field_validation_depth.md` |
| Field reference | `docs/Fields.md` |

Rows of the map and PDF figures are **depth levels**, not regions — that
is the one structural difference from the surface notebooks.

One channel, and it reuses the CLEAN square-first gradient — `KE` takes `sqrt(gradb2)` and then squares the whole expression, so the square root and the square undo each other and the field is effectively linear in `gradb2`.  No new artifact is introduced here.

## Section 1 — Setup

Everything configurable is in the next cell: the **region**, the date,
the depth levels, and the zoom size.  Change `REGION` to validate a
different part of the ocean — any key in `dbof.plotting.regions.REGIONS`
that carries a `zoom` anchor.

Default is the Gulf Stream, anchored at 60°W / 37°N — dynamically
active in every field this project computes, and the same point the
surface notebooks zoom into, so surface and depth look at the same
water.


In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"  # the only DEPTH date transferred so far
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0                  # -> a 200 x 200 km zoom box

SUBSET   = "energetics"
PIPELINE = "DEPTH"
RAW_VARS = ["Theta", "Salt"]

# Profiles (Figure 3)
N_PROFILES        = 5       # <= 5; the fixed location colours are not cycled
PROFILE_SEED      = 42      # same 5 columns for every field in the notebook
PROFILE_MAX_DEPTH = 500.0   # depth-axis limit, m; None = full 969 m column
# ------------------------------------------------------------------------

import dask
import numpy as np

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
import dbof.utils.native_gradient as NG
from dbof.preprocessing.vertical_helpers import (
    _interp_w_to_tracer_levels, _vertical_derivative,
)
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile
from dbof.global_dataset_creation.subset_definitions import (
    get_compute_fn, get_subset_definition, expand_channels_with_suffixes,
)

# tile_utils sets the Agg backend when it is imported (it writes QA PNGs
# on headless nodes), so switch back to inline AFTER the dbof imports or
# no figure in this notebook will render.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

# Channel list straight from the pipeline's own definition -- if the
# subset gains a channel, this notebook picks it up without an edit.
defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = expand_channels_with_suffixes(
    defn["compute_features_channels"], list(LEVELS),
    defn.get("extra_channels"),
)
# The mld / mld_mean strategies need MLD, which needs potential
# density.  Loading a subset without Theta/Salt fails deep inside
# mixed_layer_depth with an unhelpful AttributeError, so check here.
if {"mld", "mld_mean"} & set(LEVELS):
    _need = {"Theta", "Salt"} - set(RAW_VARS)
    assert not _need, (
        f"LEVELS includes an MLD-based level, so RAW_VARS must include "
        f"{sorted(_need)} -- MLD is derived from potential density.")

print(f"subset   : {SUBSET}")
print(f"channels : {CHANNELS}")

## Section 2 — Load the tile and compute the fields

We do **not** run `generate-global` here.  That would compute the whole
planet in order to look at one place.

Instead this notebook works on **one tile** — the 720 × 720 × 51 block
the `dbof.tiles` workflow already defines: one LLC face, the full water
column, about 1400 × 1400 km, centred on the region's anchor.  A tile is
*exactly one chunk* of the depth store, so loading it costs one S3 GET
per variable (~106 MB per 3D field).  Tiles are 720-aligned and faces
are 6 × 720 wide, so a tile can never straddle two faces.

Then the **production** compute function for this subset runs on it,
and internally applies the four depth strategies.  Same code as
production, one tile's worth of data.

One thing this costs us: the tile's xgcm grid has **no face
connections**, so cells near the boundary have no neighbours and their
horizontal gradients are wrong.  That rim is NaN'd, using the per-field
widths `tiles/field_registry.py` already records (0 for purely vertical
fields, 1 for staggered interpolation, 3 for gradient and Jacobian
chains).

**A tile samples the region, it does not cover it.**  "Gulf Stream"
here means the ~1400 km tile around 60°W / 37°N — not the whole
80–40°W box the surface notebooks use as a row.


In [ ]:
# Anchor -> rect pixel -> the tile that contains it.
S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)

i_rect, j_rect = tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3)
tile = rect_ij_to_tile(i_rect, j_rect)
print(f"region : {REGION} anchored at ({ANCHOR_LON}, {ANCHOR_LAT})")
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}, "
      f"j={tile.j_face_slice}, i={tile.i_face_slice}")

# Load the tile + its grid, then merge and build a LOCAL xgcm grid.
ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(S3, DATE, tile, RAW_VARS)
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)
print(f"extent : lon [{XC.min():.2f}, {XC.max():.2f}], "
      f"lat [{YC.min():.2f}, {YC.max():.2f}], "
      f"land {100 * LAND.mean():.1f}%")

### The finals, and the intermediates the figures need

`get_compute_fn("DEPTH", SUBSET)` is the production entry point — the
same callable `generate-global` dispatches to — so the finals below are
the pipeline's own numbers.

The **intermediates** are a different matter: the global products never
store them, so they are recomputed here from the same `ds_merge` the
finals came from.  That is deliberate — it means each figure's chain
shows the actual steps, not a reconstruction.


In [ ]:
store = get_compute_fn(PIPELINE, SUBSET)(ds_merge, xgrid, CHANNELS)
print(f"computed : {sorted(store)}")

mld = CFAD.mixed_layer_depth(ds_merge)

_gb2 = CF.grad_b2(ds_merge, xgrid)
_f3d = CF.coriolis_parameter(ds_merge, xgrid)
# KE is dispatcher-inline; reproduced here so it can be profiled in 3D.
_KE = 0.5 * (mld * np.sqrt(_gb2) / _f3d) ** 2

PROFILE_3D = {
    "b": CF.buoyancy_of_field(ds_merge),
    "gradb2": _gb2,
    "KE": _KE,
}
# Fields the production call already reduced to levels: keep them in 3D
# for the profiles, but do not recompute their level slices.
STORE_BASES = {dfig.channel_base(k, LEVELS) for k in store}
live = dfig.compute_levels(
    {k: v for k, v in PROFILE_3D.items() if k not in STORE_BASES},
    ds_merge, mld=mld, levels=LEVELS)

# Both 2D by nature -- no depth profile.
live["mixed_layer_depth"] = mld
live["coriolis_f"] = _f3d
live = dict(zip(live, dask.compute(*live.values(), retries=10)))

In [ ]:
# How wide the invalid rim is for this subset, straight from the tile
# registry (0 here: nothing in this chain takes a horizontal gradient).
EDGE_MARGIN = dfig.edge_margin_for(
    list(defn["compute_features_channels"])
    + list(defn.get("extra_channels") or []))

# NaN that rim, mask land with the surface hFacC (what production does),
# and reshape into the {base: {level: (x, y, arr)}} the figures take.
level_arrays = dfig.pack_tile_levels(
    {**live, **store}, XC, YC, edge_margin=EDGE_MARGIN,
    land_mask=LAND, levels=LEVELS)

In [ ]:
# Five ocean columns, seeded and spread across the tile, reused by every
# field in this notebook so the profile panels are comparable.
POINTS = dfig.pick_profile_points(
    LAND, n=N_PROFILES, edge_margin=max(EDGE_MARGIN, 1),
    seed=PROFILE_SEED)

# Full water column at those five points -- a few hundred numbers per
# field, so this is cheap next to the maps.
PROFILES, DEPTH_M = dfig.sample_profiles(PROFILE_3D, ds_merge, POINTS)

# MLD at each point, to mark on the profiles.
MLD_AT_POINTS = (
    [level_arrays["mixed_layer_depth"]["sfc"][2][j, i] for j, i in POINTS]
    if "mixed_layer_depth" in level_arrays else None)

## Section 3 — Subset: `energetics`

| Channel | Kind |
|---|---|
| `KE_{sfx}` | base × depth suffixes |

A single output channel, built from three fields validated elsewhere:
MLD (`stratification.ipynb`), |∇b|² (`frontal_structure.ipynb`) and f
(`kinematic.ipynb`).


## Section 4 — Field & dependency table

| FIELD | UNITS | EQUATION | DEPENDS ON | CODE |
|---|---|---|---|---|
| `b` | m s⁻² | b = g·σ₀/ρ₀ | Theta, Salt | `calculate_fields.buoyancy_of_field` |
| `gradb2` | s⁻⁴ | \|∇b\|², squared BEFORE interp | b | `native_gradient.calculate_grad_squared_tracer` |
| `KE_{sfx}` | m² s⁻² | KE = ½·(MLD·\|∇b\|/f)² | gradb2, MLD, f | `depth_subsets.compute_energetics` (inline) |

**No new artifact.**  The implementation computes
`sqrt(grad_b2)` and then squares the whole bracket, so the square root
and the outer square cancel: KE is effectively `½·(MLD/f)²·gradb2`,
linear in the CLEAN square-first gradient.  Nothing is squared after an
interpolation.

Two real caveats that are not artifacts:

- KE is an **eddy-scaling estimate**, not a diagnosed kinetic energy.
  It is a mixed-layer instability scaling, so it is only meaningful
  where a mixed layer exists and a front is present.
- It is f-normalised (f in the denominator, squared), so it diverges
  near the equator.  On an equatorial tile this field is meaningless.

**Note:** the equation lives inline in the subset dispatcher rather
than in `calculate_fields_at_depth`.  The original plan proposed
promoting it; that has not been done.

**Tile edge rim:** `edge_margin = 3` (the gradient chain).


## Section 5 — Per-field validation

Three figures per field.

**Figure 1 — maps.**  Columns are the dependency chain, raw → final.
Rows are the four depth levels over the whole tile, then the same four
zoomed to a 200 × 200 km box; the crimson square on the whole-tile rows
is where the zoom is.  One colour scale per column, shared by every row
including the zooms, so nothing changes colour when you look closer.

Fields that **do not vary with depth** get two rows instead of eight —
whole tile and zoom.  `mixed_layer_depth` and `ml_heat_content` are
both integrals over the entire water column, so four identical depth
rows would say nothing.  Their 3D chain inputs are shown at the surface
in those figures, and the title says so.

**Figure 2 — PDFs.**  Same columns; four rows, the whole tile at each
level.  Bins are shared down a column, so reading a column top to
bottom shows how the distribution changes with depth.  The zoom boxes
are deliberately absent — too few cells to make an honest histogram.

**Figure 3 — profiles.**  Five ocean columns, spread across the tile
and fixed by a seed so every field profiles the same water.  The
leftmost panel shows where they are, as numbered colour-coded ×; then
one panel per 3D field in the chain, with the **surface at the top and
depth increasing downward**.  Dashed horizontal lines mark each
location's mixed-layer depth.  The location numbers repeat in the
legend, so the five are distinguishable without relying on colour.

("Grid" in the function names below means the rows × columns array of
panels — not the model's Arakawa C-grid, which is `docs/Grid.md`.)


In [ ]:
# Section 5 helpers: one call per figure, shared by every field.
CHAINS = {
    "KE": ["b", "gradb2", "mixed_layer_depth", "coriolis_f", "KE"],
}
LOG_FIELDS = {"KE", "gradb2"}

# Fields with no depth dependence -- integrals over the whole column.
# Their figures collapse to 2 rows (whole tile + zoom) / 1 PDF row.
DEPTH_INVARIANT = {"coriolis_f", "mixed_layer_depth"}


def figure1_maps(field):
    """Figure 1: chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    note = (" | depth-invariant: 3D inputs shown at the surface"
            if flat else "")
    dfig.depth_map_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        region=REGION,
        levels=("sfc",) if flat else LEVELS,
        row_labels=(("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom")
                    if flat else None),
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        zoom_half_km=ZOOM_HALF_KM,
        suptitle=(f"Figure 1 — {field} | {REGION} tile | columns = "
                  f"dependency chain, rows = depth{note}"),
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: PDFs, chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    dfig.depth_pdf_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        levels=("sfc",) if flat else LEVELS,
        row_labels=("whole tile",) if flat else None,
        log10_fields=LOG_FIELDS,
        suptitle=(f"Figure 2 — {field} | {REGION} tile | density; "
                  f"land + rim NaNs dropped; bins shared down each column"),
    )
    plt.show()


def figure3_profiles(field):
    """Figure 3: depth profiles at the five fixed locations."""
    dfig.depth_profile_grid(
        CHAINS[field], PROFILES, DEPTH_M, CMAP_CFG,
        points=POINTS, level_arrays=level_arrays, region=REGION,
        mld_at_points=MLD_AT_POINTS,
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        max_depth=PROFILE_MAX_DEPTH,
        suptitle=(f"Figure 3 — {field} | {REGION} tile | profiles at "
                  f"{len(POINTS)} locations; surface at top"),
    )
    plt.show()

In [ ]:
# Safety net: every field named in a chain must actually have been
# computed, or the figure call fails deep inside matplotlib.
_missing = sorted({f for c in CHAINS.values() for f in c}
                  - set(level_arrays))
assert not _missing, f"chain fields never computed: {_missing}"
print(f"chains OK : {len(CHAINS)} fields, "
      f"{len({f for c in CHAINS.values() for f in c})} distinct columns")

### KE — mixed-layer eddy kinetic energy scaling

**KE = ½·(MLD·|∇b|/f)²**  [m² s⁻²]

The energy scale of mixed-layer instabilities.  Largest where a deep mixed layer coincides with a sharp front — on the Gulf Stream tile, the north side of the jet.  Because it is quadratic in both MLD and |∇b|, it is very heavy-tailed; the log colour scale is essential.  Rows differ only through |∇b| at each level, since MLD and f have no depth dependence.

In [ ]:
figure1_maps("KE")

In [ ]:
figure2_pdfs("KE")

In [ ]:
figure3_profiles("KE")

## Section 6 — Literature comparison

**PENDING — nothing to build here yet.**

The comparison figure is chosen *after* the literature figure is, not
before.  Once LH picks a paper figure and drops the PNG into
`../literature_figures/` (naming convention
`{field(s)}_{Citation}_{description}.png`), we decide which of our
panels belongs beside it and add a subsection here — one subsection per
reference, using `dbof.plotting.literature_comparison.side_by_side`.

Leave this section as-is until then.


## Summary — did every channel come out sane?

Coverage and range for each channel at each level, then the physical
checks that are worth failing loudly on.


In [ ]:
# Coverage + range per field per level.
print(f"{'field':<22}{'level':<10}{'finite %':>9}"
      f"{'min':>14}{'max':>14}")
print("-" * 69)
for field in sorted(level_arrays):
    for lev in LEVELS:
        arr = level_arrays[field][lev][2]
        finite = np.isfinite(arr)
        pct = 100.0 * finite.mean()
        lo = np.nanmin(arr) if finite.any() else np.nan
        hi = np.nanmax(arr) if finite.any() else np.nan
        print(f"{field:<22}{lev:<10}{pct:>8.1f}%{lo:>14.4g}{hi:>14.4g}")

In [ ]:
# Physical checks.  These assert -- a red cell here is a real problem.
ke = level_arrays["KE"]["sfc"][2]
gb2 = level_arrays["gradb2"]["sfc"][2]
mld_a = level_arrays["mixed_layer_depth"]["sfc"][2]
f_a = level_arrays["coriolis_f"]["sfc"][2]

CHECKS = [
    ("KE non-negative (it is a square)",
     np.nanmin(ke) >= 0,
     f"min = {np.nanmin(ke):.3e} m2/s2"),
    ("KE reproduces 0.5*(MLD*sqrt(gradb2)/f)**2 from its own columns",
     np.allclose(ke, 0.5 * (mld_a * np.sqrt(gb2) / f_a) ** 2,
                 rtol=1e-3, equal_nan=True),
     "recomputed from the chain columns"),
    ("KE is heavy-tailed (quadratic in two heavy-tailed inputs)",
     np.nanpercentile(ke, 99) / max(np.nanmedian(ke), 1e-30) > 20.0,
     f"p99/median = "
     f"{np.nanpercentile(ke, 99) / max(np.nanmedian(ke), 1e-30):.0f}"),
    ("KE in a plausible range (< 10 m2/s2)",
     np.nanpercentile(ke, 99.9) < 10.0,
     f"p99.9 = {np.nanpercentile(ke, 99.9):.3f} m2/s2"),
    ("KE finite over most of the tile",
     np.isfinite(ke).mean() > 0.5,
     f"{100 * np.isfinite(ke).mean():.1f}% finite"),
]

failures = []
for name, ok, detail in CHECKS:
    print(f"{'OK  ' if ok else 'FAIL'}  {name}  ({detail})")
    if not ok:
        failures.append(name)
assert not failures, f"physical checks failed: {failures}"
print("\nAll physical checks passed.")

---

### Cross-references

- **MLD** — `stratification.ipynb`.  KE is quadratic in it, so MLD
  errors are doubled here.
- **|∇b|²** — `frontal_structure.ipynb`; the square-first form is why
  this subset introduces no new artifact.
- **f** — `kinematic.ipynb`.
